# Nested Tensors Are Cool

> disposable cameras are cool, but i dont really understand the point

In [ ]:
#| default_exp nested

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch
from tqdm import tqdm

In [ ]:
#| export
def return_nested_dim(x):
    for i, d in enumerate(x.shape):
        if isinstance(d, torch.SymInt):
            return i
    return None

def flatten_dim_to_batch(x, dim=1):
    reshaped_list = []
    nested_dim = return_nested_dim(x)
    assert nested_dim is not None, "x is not nested"
    if dim != 1:
        x = x.transpose(dim, 1)
    for x_i in x:
        patch_tensors = list(x_i.unbind(0))
        reshaped_list.extend(patch_tensors)
    if nested_dim == dim:
        # there is no more nesting, so we can stack
        x = torch.stack(reshaped_list, dim=0) # [new bs x transformer_seq_len x d_model]
    else:
        x = torch.nested.as_nested_tensor(reshaped_list, layout=torch.jagged)
    return x

def unflatten_dim_from_batch(embs, c_in):
    bs = embs.size(0) // c_in
    restored_list = [
                    torch.stack([
                        embs[i * c_in + c] 
                        for c in range(c_in)
                    ], dim=0).transpose(0, 1)  # Stack channels, then transpose
                    for i in range(bs)
                ]
    embs = torch.nested.as_nested_tensor(restored_list, layout=torch.jagged)
    embs = embs.transpose(1, 2)  
    return embs

In [ ]:
#| export
def coerce_offsets(src, tgt):
    """
    See https://github.com/pytorch/pytorch/issues/138180
    Coerce offsets to be the same for two nested tensors so element wise operations can occur.

    use: src = coerce_offsets(src, tgt)
    """
    assert torch.eq(src.offsets(), tgt.offsets()).all().item()
    assert src._ragged_idx == tgt._ragged_idx

    def mb_get_size(t):
        return t.shape[0] if t is not None else None

    return torch.nested.nested_tensor_from_jagged(
        src.values(),
        tgt.offsets(),
        None,
        src._ragged_idx,
        mb_get_size(src._max_seqlen_tensor) if tgt._max_seqlen_tensor is None else mb_get_size(src._max_seqlen_tensor),
        mb_get_size(src._min_seqlen_tensor) if tgt._min_seqlen_tensor is None else mb_get_size(src._min_seqlen_tensor),
    )

In [ ]:
#| export
def get_embeddings_nested(data_loader, model, dataloader_name="", autocast=True):
    embs, targets, times = [], [], []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model = model.eval()
    with torch.no_grad():
        for batch in tqdm(data_loader, desc=f"Retrieving embeddings {dataloader_name}"):
            x, y, time = batch
            x = x.to(device)
            if autocast:
                with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
                    emb, *_ = model.predict_step((x,y, time), 0)
                    emb = emb.mean(dim=2) # mean pool over patches


            else:
                emb, *_ = model.predict_step((x,y, time), 0)
                emb = emb.mean(dim=2) # mean pool over patches
            embs.append(emb.cpu())
            targets.append(y.cpu())
            times.append(time.cpu())
    return embs, targets, times

def get_predictions_nested(data_loader, model, dataloader_name="", autocast=True):
    preds, targets = [], []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model = model.eval()
    with torch.no_grad():
        for batch in tqdm(data_loader, desc=f"Predicting {dataloader_name}"):
            x, y, *_ = batch
            x = x.to(device)
            y = y.to(device)
            if autocast:
                with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
                    pred,_ = model.predict_step((x,y), 0)
            else:
                pred,_ = model.predict_step((x,y), 0)
            preds.append(pred.cpu())
            targets.append(y.cpu())
    return preds, targets

def get_predictions_nested_survival(data_loader, model, dataloader_name="", autocast=True):
    preds, targets, times, demographics = [], [], [], []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model = model.eval()
    with torch.no_grad():
        for batch in tqdm(data_loader, desc=f"Predicting {dataloader_name}"):
            batch = tuple([b.to(device) for b in batch])
            if autocast:
                with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
                    pred,y,time,*demographic_targets = model.predict_step(batch, 0)
            else:
                pred,y,time,*demographic_targets = model.predict_step(batch, 0)
            preds.append(pred.cpu())
            targets.append(y.cpu())
            times.append(time.cpu())
            if len(demographic_targets) > 0:
                demographics.append(demographic_targets[0].cpu())
    if len(demographics) > 0:
        return preds, targets, times, demographics
    else:
        return preds, targets, times

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()